In [ ]:
!pip install transformers accelerate torch sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 47.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 83.6 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitlin

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_id = "microsoft/Phi-3-mini-4k-instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

# Upload your processed_papers.json
from google.colab import files
uploaded = files.upload()

# Load the data
import json
import io

papers = json.load(io.BytesIO(uploaded['processed_papers.json']))
papers_subset = papers[:10]  # Take first 10 papers

# Process function
def process_paper(paper_text, filename):
    prompt = f"""<|system|>
You are a research assistant specializing in cotton agriculture. Analyze this research paper and extract key findings.
</|system|>

<|user|>
Paper: {filename}
Content: {paper_text[:3000]}
Provide a concise summary of the key findings and their importance.
</|user|>
"""

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=300,
        temperature=0.5,
        do_sample=True
    )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Remove the prompt from the response
    response = response.replace(prompt, "").strip()
    return response

# Process all 10 papers
for i, paper in enumerate(papers_subset):
    print(f"Processing paper {i+1}/10: {paper['filename']}")
    paper['summary'] = process_paper(paper['cleaned_text'], paper['filename'])

    # Save after each paper as backup
    with open(f'partial_results_{i+1}.json', 'w') as f:
        json.dump(papers_subset[:i+1], f, indent=2)
    print(f"Saved progress for {i+1} papers")

# Save final results
with open('results_10papers.json', 'w') as f:
    json.dump(papers_subset, f, indent=2)

# Download the results
files.download('results_10papers.json')

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Saving processed_papers.json to processed_papers.json
Processing paper 1/10: 10.2135_ccropsci1982.0011183X002200040019x.pdf
Saved progress for 1 papers
Processing paper 2/10: 10.2135_cropsci1961.0011183X000100030004x.pdf
Saved progress for 2 papers
Processing paper 3/10: 10.2135_cropsci1961.0011183X000100050003x.pdf
Saved progress for 3 papers
Processing paper 4/10: 10.2135_cropsci1961.0011183X000100050020x.pdf
Saved progress for 4 papers
Processing paper 5/10: 10.2135_cropsci1961.0011183X000100060001x.pdf
Saved progress for 5 papers
Processing paper 6/10: 10.2135_cropsci1961.0011183X000100060005x.pdf
Saved progress for 6 papers
Processing paper 7/10: 10.2135_cropsci1961.0011183X000100060026x.pdf
Saved progress for 7 papers
Processing paper 8/10: 10.2135_cropsci1962.0011183X000200010014x.pdf
Saved progress for 8 papers
Processing paper 9/10: 10.2135_cropsci1962.0011183X000200010019x.pdf
Saved progress for 9 papers
Processing paper 10/10: 10.2135_cropsci1962.0011183X000200010020x.pdf
Sa

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>